# Documentação da base de tráfego

**Etapa:** definição do contrato temporal  
**Responsável:** Bruna Carvalho Cardoso · Grupo 5 (PLS) · Base 2

Este notebook registra fonte, unidade, período, frequência, horizonte, disponibilidade das externas e o recorte congelado. O texto completo está em `DOCUMENTACAO_BASE2.md` (mesmo estilo da documentação do ouro e da Jena).

**Entrada:** `bases/grupo2/Metro_Interstate_Traffic_Volume.csv`  
**Resultado:** contrato único para os quatro modelos

| Item | Valor |
|---|---|
| Alvo | `traffic_volume` (veículos/hora) |
| Fonte | UCI Metro Interstate Traffic Volume |
| Período bruto | 2012-10-02 → 2018-09-30 |
| Frequência | horária (com duplicatas e lacunas) |
| Horizonte | 1 hora |
| `random_state` | 67 |
| Split | 80/20 cronológico |

Calendário e feriado são conhecidos antecipadamente. Temperatura/chuva/neve/nuvens **observadas em t+1** são vazamento; usamos apenas lag 1.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
raw = data_lib.load_raw()
print("shape:", raw.shape)
print("período:", raw.date_time.min(), "→", raw.date_time.max())
print("timestamps duplicados:", raw.date_time.duplicated().sum())
display(raw.head())
display(raw.describe(include="all").T)


shape: (48204, 9)
período: 2012-10-02 09:00:00 → 2018-09-30 23:00:00
timestamps duplicados: 7629


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
holiday,48204,12,None,48143,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temp,48204.0,NaN,NaN,NaN,281.20587,0.0,272.16,282.45,291.806,310.07,13.338232
rain_1h,48204.0,NaN,NaN,NaN,0.334264,0.0,0.0,0.0,0.0,9831.3,44.789133
snow_1h,48204.0,NaN,NaN,NaN,0.000222,0.0,0.0,0.0,0.0,0.51,0.008168
clouds_all,48204.0,NaN,NaN,NaN,49.362231,0.0,1.0,64.0,90.0,100.0,39.01575
weather_main,48204,11,Clouds,15164,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weather_description,48204,38,sky is clear,11665,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_time,48204,NaN,NaN,NaN,2016-01-05 10:46:16.773711616,2012-10-02 09:00:00,2014-02-06 11:45:00,2016-06-11 03:30:00,2017-08-11 06:00:00,2018-09-30 23:00:00,NaN
traffic_volume,48204.0,NaN,NaN,NaN,3259.818355,0.0,1193.0,3380.0,4933.0,7280.0,1986.86067
